<a href="https://colab.research.google.com/github/Chamodya07/crimesense/blob/main/CrimeSense_TabularOnly_XGBoost_GPU_FIXED_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install xgboost


In [2]:
from xgboost import XGBClassifier
# ============================================
# 1) Environment + Dependencies (Colab-friendly)
# ============================================
import sys, platform, os, warnings
warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

# Core
import numpy as np
import pandas as pd

# ML
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt

# SHAP (install if missing)
try:
    import shap
except Exception:
    !pip -q install shap
    import shap

print("✅ Imports OK")


Python: 3.12.12
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
✅ Imports OK


In [3]:
# === 3) Setup Folders (Google Colab) ===
import os

# Google Colab uses /content as the default working directory
BASE = "/content/crimesense"
print("🌐 Running on Google Colab")

# Create all required subdirectories
for d in ["data/raw", "data/curated", "models", "reports"]:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)

RAW = os.path.join(BASE, "data", "raw")
CUR = os.path.join(BASE, "data", "curated")

print(f"✅ BASE: {BASE}")
print(f"✅ RAW:  {RAW}")
print(f"✅ CUR:  {CUR}")

🌐 Running on Google Colab
✅ BASE: /content/crimesense
✅ RAW:  /content/crimesense/data/raw
✅ CUR:  /content/crimesense/data/curated


In [4]:
# === 4) Upload Kaggle Credentials (kaggle.json) ===
from google.colab import files
import os

kaggle_path = "/root/.kaggle/kaggle.json"

if os.path.exists(kaggle_path):
    print("✅ Kaggle already configured")
else:
    print("📤 Please upload your kaggle.json file:")
    uploaded = files.upload()

    !mkdir -p /root/.kaggle
    !mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json
    print("✅ Kaggle configured!")


📤 Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
✅ Kaggle configured!


In [5]:
# === 5) Download Datasets ===
import glob
import zipfile

RAW = f"{BASE}/data/raw"

# Download crime datasets WITH NARRATIVE TEXT
print("📥 Downloading datasets...\n")

# ======= ALTERNATIVE DATASETS WITH TEXT COLUMNS =======
# Since SF Crime competition is restricted, using these alternatives:

# 1. Chicago Crime - Has Description column with incident details
print("🔹 Chicago Crime (has Description column)...")
!kaggle datasets download -d salikhussaini49/chicago-crimes-data -p {RAW} -q

# 2. LA Crime - Has Crm Cd Desc column
print("🔹 LA Crime (has Crm Cd Desc)...")
!kaggle datasets download -d venkatsairo4899/los-angeles-crime-data-2020-2023 -p {RAW} -q

# 3. Alternative: Austin Crime Reports (has Category Description)
print("🔹 Austin Crime (has offense descriptions)...")
!kaggle datasets download -d adityakadiwal/austin-crime-report -p {RAW} -q 2>/dev/null || echo "   Austin dataset unavailable"

# 4. Alternative: Seattle Crime (has Crime Against Category, Offense)
print("🔹 Seattle Crime...")
!kaggle datasets download -d sohier/seattle-crime -p {RAW} -q 2>/dev/null || echo "   Seattle dataset unavailable"

# Unzip all
print("\n📦 Unzipping...")
for z in glob.glob(f"{RAW}/*.zip"):
    try:
        with zipfile.ZipFile(z, 'r') as zf:
            zf.extractall(RAW)
        print(f"  ✅ {os.path.basename(z)}")
    except:
        print(f"  ⚠️ Bad zip: {os.path.basename(z)}")

# List files
print(f"\n📁 Downloaded CSV files:")
csv_files = glob.glob(f"{RAW}/*.csv")
if csv_files:
    for csv in csv_files[:5]:  # Show first 5
        size_mb = os.path.getsize(csv) / (1024 * 1024)
        print(f"  - {os.path.basename(csv):<40} ({size_mb:.1f} MB)")
    if len(csv_files) > 5:
        print(f"  ... and {len(csv_files)-5} more files")
else:
    print("  ⚠️ No CSV files found")

# ======= VERIFY TEXT COLUMNS =======
import pandas as pd
print("\n" + "="*60)
print("📊 VERIFYING NARRATIVE TEXT COLUMNS")
print("="*60)

text_cols_found = {}
for csv in glob.glob(f"{RAW}/*.csv")[:10]:  # Check first 10 files
    try:
        df_check = pd.read_csv(csv, nrows=10)
        # Look for text/narrative columns
        text_candidates = [c for c in df_check.columns if any(x in c.lower() for x in
            ['descript', 'description', 'narrative', 'desc', 'crm_cd_desc', 'offense', 'incident'])]
        if text_candidates:
            # Check if column has actual text (not just codes)
            for col in text_candidates:
                sample_text = df_check[col].dropna().astype(str).head(3)
                avg_len = sample_text.str.len().mean()
                if avg_len > 10:  # At least 10 chars average
                    text_cols_found[os.path.basename(csv)] = {
                        'column': col,
                        'avg_length': int(avg_len),
                        'sample': sample_text.iloc[0][:60] if len(sample_text) > 0 else "N/A"
                    }
                    print(f"✅ {os.path.basename(csv)}")
                    print(f"   Column: '{col}' (avg {int(avg_len)} chars)")
                    print(f"   Sample: \"{sample_text.iloc[0][:60]}...\"" if len(sample_text) > 0 else "")
                    break
    except Exception as e:
        pass

if text_cols_found:
    print(f"\n🎯 Found {len(text_cols_found)} files with good text columns!")
else:
    print("\n⚠️ No substantial text columns found")
    print("   Will use synthetic narratives (time + location based)")

📥 Downloading datasets...

🔹 Chicago Crime (has Description column)...
Dataset URL: https://www.kaggle.com/datasets/salikhussaini49/chicago-crimes-data
License(s): apache-2.0
🔹 LA Crime (has Crm Cd Desc)...
Dataset URL: https://www.kaggle.com/datasets/venkatsairo4899/los-angeles-crime-data-2020-2023
License(s): U.S. Government Works
🔹 Austin Crime (has offense descriptions)...
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/adityakadiwal/austin-crime-report
   Austin dataset unavailable
🔹 Seattle Crime...
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/sohier/seattle-crime
   Seattle dataset unavailable

📦 Unzipping...
  ✅ los-angeles-crime-data-2020-2023.zip
  ✅ chicago-crimes-data.zip

📁 Downloaded CSV files:
  - Chicago_Crimes.csv                       (1903.8 MB)
  - Crime_Data_from_2020_to_Present.csv      (184.5 MB)

📊 VERIFYING NARRATIVE TEXT COLUMNS
✅ Chicago_Crimes.csv
   Column: 'description' (avg 16 ch

In [6]:
# === Define RAW_DIR (fix for NameError) ===
import os
import numpy as np
import pandas as pd

# If BASE exists, use it; otherwise use default colab path
try:
    RAW_DIR = f"{BASE}/data/raw"
except NameError:
    RAW_DIR = "/content/data/raw"

os.makedirs(RAW_DIR, exist_ok=True)

print("RAW_DIR =", RAW_DIR)
print("Exists?", os.path.exists(RAW_DIR))
print("CSV files:", [f for f in os.listdir(RAW_DIR) if f.lower().endswith(".csv")][:10])

# ============================================
# 3) Normalization: map different city schemas -> one unified schema
# ============================================
from typing import Optional, Dict

def first_existing(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

VIOLENT_TYPES = {
    "HOMICIDE", "MURDER", "CRIM SEXUAL ASSAULT", "CRIMINAL SEXUAL ASSAULT",
    "ROBBERY", "ASSAULT", "BATTERY", "KIDNAPPING", "ARSON",
    "RAPE", "FELONY ASSAULT", "MURDER & NON-NEGL. MANSLAUGHTER",
    "CRIMINAL HOMICIDE", "FORCIBLE RAPE"
}

def normalize_incidents(df: pd.DataFrame, city_hint: str="Unknown") -> pd.DataFrame:
    df = df.copy()

    # --- city
    city = city_hint

    # --- date/time
    date_col = (first_existing(df, [c for c in df.columns if "date" in c.lower()])
                or first_existing(df, ["Dates", "DATE OCC", "CMPLNT_FR_DT", "CMPLNT_TO_DT"]))
    time_col = (first_existing(df, [c for c in df.columns if "time" in c.lower()])
                or first_existing(df, ["TIME OCC", "CMPLNT_FR_TM"]))

    # Parse date
    if date_col:
        df["date"] = pd.to_datetime(df[date_col], errors="coerce")
    else:
        df["date"] = pd.NaT

    # Extract hour
    if time_col and time_col in df.columns:
        t = df[time_col].astype(str).str.replace(":", "", regex=False).str.strip()
        t = t.str.zfill(4)
        df["hour"] = pd.to_numeric(t.str[:2], errors="coerce")
    else:
        df["hour"] = df["date"].dt.hour

    # --- primary type / offense
    primary = first_existing(df, ["Primary Type", "OFNS_DESC", "Crm Cd Desc", "CRM_CD_DESC",
                                  "OFFENSE_DESCRIPTION", "crime_type", "primary_type"])
    df["primary_type"] = df[primary].astype(str).str.upper().str.strip() if primary else "UNKNOWN"

    # --- location type
    loc = first_existing(df, ["Location Description", "PREM_DESC", "PREMISE_DESC",
                              "LOC_OF_OCCUR_DESC", "location_description", "premise"])
    df["location_desc"] = df[loc].astype(str).str.upper().str.strip() if loc else "UNKNOWN"

    # --- weapon
    weapon = first_existing(df, ["Weapon Description", "WEAPON_DESC", "Weapon Used Cd",
                                 "Weapon Desc", "weapon_desc"])
    df["weapon_desc"] = df[weapon].astype(str).str.upper().str.strip() if weapon else "NONE"

    # --- arrest
    arrest = first_existing(df, ["Arrest", "ARREST_KEY", "Is Arrested", "arrest"])
    if arrest:
        a = df[arrest]
        if a.dtype == bool:
            df["arrest"] = a.astype(int)
        else:
            df["arrest"] = a.astype(str).str.upper().isin(["TRUE","T","YES","Y","1"]).astype(int)
    else:
        df["arrest"] = np.nan

    # --- domestic
    domestic = first_existing(df, ["Domestic", "domestic"])
    if domestic:
        d = df[domestic]
        if d.dtype == bool:
            df["domestic"] = d.astype(int)
        else:
            df["domestic"] = d.astype(str).str.upper().isin(["TRUE","T","YES","Y","1"]).astype(int)
    else:
        df["domestic"] = np.nan

    # --- lat/lon
    lat = first_existing(df, ["Latitude", "LAT", "LATITUDE", "lat"])
    lon = first_existing(df, ["Longitude", "LON", "LONGITUDE", "lon", "lng"])
    df["latitude"]  = pd.to_numeric(df[lat], errors="coerce") if lat else np.nan
    df["longitude"] = pd.to_numeric(df[lon], errors="coerce") if lon else np.nan

    # Derived helpers
    df["city"] = city
    df["is_night"] = df["hour"].between(20, 23) | df["hour"].between(0, 5)

    keep = ["city","date","hour","is_night","primary_type","location_desc","weapon_desc",
            "arrest","domestic","latitude","longitude"]
    return df[keep]

def load_and_normalize_all(raw_dir: str) -> pd.DataFrame:
    rows = []
    for fn in sorted(os.listdir(raw_dir)):
        if not fn.lower().endswith(".csv"):
            continue
        path = os.path.join(raw_dir, fn)
        low = fn.lower()

        if "chicago" in low:
            city = "Chicago"
        elif "nypd" in low or "nyc" in low:
            city = "NYC"
        elif low.startswith("la_") or "los" in low or "angeles" in low:
            city = "LA"
        else:
            city = "Unknown"

        print("📄 Loading:", fn, "| city_hint:", city)
        df0 = pd.read_csv(path, low_memory=False)
        rows.append(normalize_incidents(df0, city_hint=city))

    if not rows:
        raise FileNotFoundError(f"No CSV files found in {raw_dir}. Upload your datasets first.")

    return pd.concat(rows, ignore_index=True)

df = load_and_normalize_all(RAW_DIR)
print("✅ Unified rows:", len(df))
df.head()


RAW_DIR = /content/crimesense/data/raw
Exists? True
CSV files: ['Chicago_Crimes.csv', 'Crime_Data_from_2020_to_Present.csv']
📄 Loading: Chicago_Crimes.csv | city_hint: Chicago
📄 Loading: Crime_Data_from_2020_to_Present.csv | city_hint: Unknown
✅ Unified rows: 8808682


,city,date,hour,is_night,primary_type,location_desc,weapon_desc,arrest,domestic,latitude,longitude
0,Chicago,2017-11-20 17:05:00+00:00,17,False,BATTERY,DEPARTMENT STORE,NONE,1.0,0.0,NaN,NaN
1,Chicago,2017-08-31 20:42:00+00:00,20,True,PUBLIC PEACE VIOLATION,RESTAURANT,NONE,0.0,0.0,NaN,NaN
2,Chicago,2017-05-16 14:50:00+00:00,14,False,DECEPTIVE PRACTICE,BANK,NONE,0.0,0.0,NaN,NaN
3,Chicago,2017-05-01 17:00:00+00:00,17,False,DECEPTIVE PRACTICE,BANK,NONE,0.0,0.0,NaN,NaN
4,Chicago,2017-05-18 19:35:00+00:00,19,False,PROSTITUTION,HOTEL/MOTEL,NONE,1.0,0.0,NaN,NaN


In [7]:
# ============================================
# 4) Create Targets (Proxy Labels) for Multi-Output Training
#    - risk_level: Low/Medium/High
#    - crime_severity: Violent/Non-violent
#    - offender_experience: Low/Medium/High
# ============================================
# IMPORTANT:
# Many public datasets do NOT contain direct labels like 'risk level' or 'offender experience'.
# So we create *proxy labels* from structured columns for training/demo.

def severity_label(primary_type: str) -> str:
    pt = str(primary_type).upper()
    return "Violent" if any(v in pt for v in VIOLENT_TYPES) else "Non-violent"

df["crime_severity"] = df["primary_type"].apply(severity_label)

# Simple structured risk score (rule-based) -> then binned for classification label
risk_score = (
    (df["crime_severity"] == "Violent").astype(int) * 2
    + (df["weapon_desc"].fillna("NONE").ne("NONE")).astype(int) * 1
    + (df["is_night"].fillna(False)).astype(int) * 1
    + (df["location_desc"].fillna("").str.contains("RESID", case=False, regex=True)).astype(int) * 1
)
# Bin into Low/Medium/High
df["risk_level"] = pd.cut(risk_score, bins=[-1,1,3,10], labels=["Low","Medium","High"]).astype(str)

# Offender experience proxy (very limited without offender history fields)
# If no priors exist, we approximate by combining violent + arrest signals
exp_score = (
    (df["crime_severity"] == "Violent").astype(int)
    + df["arrest"].fillna(0).astype(int)
)
df["offender_experience"] = pd.cut(exp_score, bins=[-1,0,1,10], labels=["Low","Medium","High"]).astype(str)

df[["primary_type","weapon_desc","is_night","location_desc","crime_severity","risk_level","offender_experience"]].head(10)


,primary_type,weapon_desc,is_night,location_desc,crime_severity,risk_level,offender_experience
0,BATTERY,NONE,False,DEPARTMENT STORE,Violent,Medium,High
1,PUBLIC PEACE VIOLATION,NONE,True,RESTAURANT,Non-violent,Low,Low
2,DECEPTIVE PRACTICE,NONE,False,BANK,Non-violent,Low,Low
3,DECEPTIVE PRACTICE,NONE,False,BANK,Non-violent,Low,Low
4,PROSTITUTION,NONE,False,HOTEL/MOTEL,Non-violent,Low,Medium
5,PROSTITUTION,NONE,False,HOTEL/MOTEL,Non-violent,Low,Medium
6,KIDNAPPING,NONE,False,GOVERNMENT BUILDING/PROPERTY,Violent,Medium,Medium
7,ASSAULT,NONE,True,CTA PLATFORM,Violent,Medium,Medium
8,ASSAULT,NONE,False,CTA BUS,Violent,Medium,Medium
9,DECEPTIVE PRACTICE,NONE,False,MOVIE HOUSE/THEATER,Non-violent,Low,Low


In [8]:
# ============================================
# 5) Train/Val/Test Split
# ============================================
# Drop rows with missing essential fields
work = df.dropna(subset=["primary_type","location_desc","weapon_desc","hour"]).copy()

FEATURES = ["city","hour","is_night","primary_type","location_desc","weapon_desc",
            "arrest","domestic","latitude","longitude"]

TARGETS = ["risk_level","crime_severity","offender_experience"]

X = work[FEATURES]
Y = work[TARGETS]

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y, test_size=0.30, random_state=42, stratify=work["crime_severity"]
)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=42, stratify=Y_temp["crime_severity"]
)

print("Train:", X_train.shape, Y_train.shape)
print("Val:  ", X_val.shape, Y_val.shape)
print("Test: ", X_test.shape, Y_test.shape)


Train: (6166077, 10) (6166077, 3)
Val:   (1321302, 10) (1321302, 3)
Test:  (1321303, 10) (1321303, 3)


In [9]:

# --- FIX: XGBoost requires numeric class labels (encode each output column) ---
from sklearn.preprocessing import LabelEncoder
import pandas as pd

label_encoders = {}
Y_train_enc = Y_train.copy()

Y_test_enc = None
if 'Y_test' in globals():
    Y_test_enc = Y_test.copy()

for col in Y_train.columns:
    le = LabelEncoder()
    Y_train_enc[col] = le.fit_transform(Y_train[col].astype(str))
    label_encoders[col] = le
    if Y_test_enc is not None:
        Y_test_enc[col] = le.transform(Y_test[col].astype(str))

print("✅ Encoded target classes per output:")
for k, le in label_encoders.items():
    print(k, list(le.classes_))


✅ Encoded target classes per output:
risk_level ['High', 'Low', 'Medium']
crime_severity ['Non-violent', 'Violent']
offender_experience ['High', 'Low', 'Medium']


In [1]:
# ============================================
# 6) Preprocess + Train (Portable XGBoost per target)
#    (NO sklearn Pipeline / ColumnTransformer)
# ============================================
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

cat_cols = ["city","primary_type","location_desc","weapon_desc"]
num_cols = ["hour","is_night","arrest","domestic","latitude","longitude"]

def build_X(df_like: pd.DataFrame) -> pd.DataFrame:
    # split to cat/num
    Xn = df_like[num_cols].copy()
    # fill missing numerics
    Xn = Xn.fillna(0)

    Xc = df_like[cat_cols].copy()
    # fill missing categoricals
    Xc = Xc.fillna("Unknown")

    # one-hot encode categoricals
    Xc_oh = pd.get_dummies(Xc, columns=cat_cols, dummy_na=False)

    return pd.concat([Xn, Xc_oh], axis=1)

# Build train/val/test matrices
X_train_mat = build_X(X_train)
X_val_mat   = build_X(X_val)
X_test_mat  = build_X(X_test)

# Freeze feature columns so inference can match
feature_cols = sorted(set(X_train_mat.columns) | set(X_val_mat.columns) | set(X_test_mat.columns))

def align_cols(Xmat: pd.DataFrame) -> pd.DataFrame:
    return Xmat.reindex(columns=feature_cols, fill_value=0)

X_train_mat = align_cols(X_train_mat)
X_val_mat   = align_cols(X_val_mat)
X_test_mat  = align_cols(X_test_mat)

# Train one model per target (portable)
models = {}
for t in TARGETS:
    y = Y_train_enc[t].astype(int).values

    clf = XGBClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        min_child_weight=1,
        gamma=0.0,
        tree_method="hist",
        device="cuda",        # works on T4
        eval_metric="logloss",
        random_state=42,
        n_jobs=0,
    )
    clf.fit(X_train_mat, y)
    models[t] = clf

print("✅ Trained models:", list(models.keys()))
print("✅ Feature columns:", len(feature_cols))

NameError: name 'X_train' is not defined

In [ ]:

# ==============================
# Helper: Decode model predictions back to original string labels (for frontend)
# ==============================
import numpy as np
import pandas as pd

def decode_multioutput_predictions(pred_array, target_columns, label_encoders):
    pred_df = pd.DataFrame(pred_array, columns=target_columns)
    for col in target_columns:
        if col in label_encoders:
            pred_df[col] = label_encoders[col].inverse_transform(pred_df[col].astype(int))
    return pred_df

# Example usage:
# preds = model.predict(X_test)   # returns numeric classes
# preds_decoded = decode_multioutput_predictions(preds, Y_train.columns, label_encoders)
# display(preds_decoded.head())


In [ ]:
# ============================================
# 7) Evaluation (Per Output Head) — encoded labels
# ============================================
Y_pred = model.predict(X_test)
Y_pred = pd.DataFrame(Y_pred, columns=TARGETS)

# Use encoded ground-truth to match numeric predictions
Y_true = Y_test_enc[TARGETS].copy()

for t in TARGETS:
    print("="*80)
    print("Target:", t)
    print(classification_report(Y_true[t].astype(int), Y_pred[t].astype(int), digits=4))

In [ ]:
# ============================================
# 8) Explainability (Global + Local SHAP)
# ============================================
# NOTE:
# With one-hot encoding, SHAP is computed on the transformed feature space.
# We'll explain the 'risk_level' head as the primary example.

# Helper to get feature names from ColumnTransformer + OneHotEncoder
def get_feature_names(prep: ColumnTransformer):
    out = []
    for name, trans, cols in prep.transformers_:
        if name == "remainder":
            continue
        if hasattr(trans, "named_steps") and "onehot" in trans.named_steps:
            ohe = trans.named_steps["onehot"]
            out.extend(list(ohe.get_feature_names_out(cols)))
        else:
            out.extend(list(cols))
    return out

prep = model.named_steps["prep"]
feat_names = get_feature_names(prep)

# Transform test data
X_test_tx = prep.transform(X_test)

# Extract the estimator for risk_level head (index 0)
risk_est = model.named_steps["clf"].estimators_[0]

# SHAP (TreeExplainer)
explainer = shap.TreeExplainer(risk_est)
# For multiclass, shap_values is a list per class
shap_values = explainer.shap_values(X_test_tx[:200])  # limit rows for speed

print("✅ SHAP ready (showing summary for risk_level)")

# Summary plot
shap.summary_plot(shap_values, features=X_test_tx[:200], feature_names=feat_names, show=False)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 9) Inference: Predict a single new case (portable)
# ============================================
import numpy as np
import pandas as pd

def case_to_matrix(case: dict) -> pd.DataFrame:
    # Build DataFrame with base FEATURES first
    base = pd.DataFrame([case])[FEATURES].copy()

    # Ensure required cols exist
    for c in FEATURES:
        if c not in base.columns:
            base[c] = 0

    # Build encoded matrix exactly like training
    Xm = build_X(base)                 # uses the function from Cell 9
    Xm = Xm.reindex(columns=feature_cols, fill_value=0)  # align to training columns
    return Xm

def predict_case(case: dict):
    Xm = case_to_matrix(case)

    pred_num = {}
    pred = {}
    for t in TARGETS:
        y_hat = int(models[t].predict(Xm)[0])
        pred_num[t] = y_hat

        # decode back to label if encoder exists
        if t in label_encoders:
            pred[t] = label_encoders[t].inverse_transform([y_hat])[0]
        else:
            pred[t] = str(y_hat)

    return pred

In [ ]:
import sklearn, joblib, numpy, pandas, sys
print("python:", sys.version)
print("sklearn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)

In [ ]:
# ============================================
# 10) Save Portable Artifacts (NO sklearn pipeline)
# ============================================
import os, json, joblib
from pathlib import Path

OUT_DIR = Path("/content/crimesense_tabular_only_portable")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save each XGBoost model as portable JSON
for t, clf in models.items():
    clf.save_model(str(OUT_DIR / f"xgb_{t}.json"))

# Save feature columns (critical)
with open(OUT_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

# Save meta (features + targets)
meta = {
    "base_features": FEATURES,
    "targets": TARGETS,
    "notes": "Portable XGBoost per-target models + get_dummies encoding."
}
with open(OUT_DIR / "model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

# Save label encoders (so you can decode)
joblib.dump(label_encoders, OUT_DIR / "label_encoders.joblib")

print("✅ Saved portable tabular artifacts to:", OUT_DIR)
print(os.listdir(OUT_DIR))